# Chest X-Ray Pneumonia (Transfer Learning Experiment )
- ### Experiments

## 1. Environment

In [ ]:
# Use Vitual Terminal (pc)
# Use Google Colab GPU (colab)

# python -m kaggle --version
# python -m kaggle auth login
# python -m kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p data

# %pip install --break-system-packages  pynvml
# %pip install --break-system-packages pandas numpy scikit-learn matplotlib seaborn plotly openpyxl pillow

In [ ]:
import psutil

memory = psutil.virtual_memory()

print(f"전체 RAM : {memory.total / 1024**3:.1f} GB")
print(f"사용 RAM : {memory.used / 1024**3:.1f} GB")
print(f"사용률   : {memory.percent}%")
print(f"여유 RAM : {memory.available / 1024**3:.1f} GB")

#### Custom Lib Link (Colab/PC)

In [2]:
## Colab 전용(기본 경로)
from google.colab import drive

drive.mount("/content/drive")

import sys

sys.path.insert(0, "/content/drive/MyDrive/AI/src")
sys.path.insert(0, "/content/drive/MyDrive/AI/projects/chest-xray-pneumonia")


Mounted at /content/drive


In [8]:
## PC 전용(기본 경로)
import sys

sys.path.insert(0, r"D:\DEV\AI\src")
sys.path.insert(0, r"D:\DEV\AI\projects\chest-xray-pneumonia")


In [1]:
## DeskTop 전용(기본 경로)
import sys

sys.path.insert( 0, "/workspace/src" ) 
sys.path.insert( 0, "/workspace/projects/chest-xray-pneumonia" )


In [ ]:
import sys

print(sys.executable)
print("\n".join(sys.path)) 

### 1.1 Import

In [3]:
 # ============================================================
# AI Development Environment Check
# ============================================================

import sys

import numpy as np
import pandas as pd
import sklearn
import torch
import torchvision
import plotly
from PIL import Image


print("=" * 60)
print("AI Development Environment")
print("=" * 60)

print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"TorchVision : {torchvision.__version__}")
print(f"Pillow      : {Image.__version__}")
print(f"Plotly      : {plotly.__version__}")


AI Development Environment
Python      : 3.12.13
NumPy       : 2.0.2
Pandas      : 2.2.2
scikit-learn: 1.6.1
PyTorch     : 2.11.0+cu128
TorchVision : 0.26.0+cu128
Pillow      : 11.3.0
Plotly      : 5.24.1


In [4]:
from cxp.cxp_config import (
    configure_environment,
)

# ============================================================
# Environment
# ============================================================

# ENV = "pc"
# ENV = "remote"
ENV = "colab"


# ============================================================
# Project Configuration
# ============================================================

config = configure_environment(ENV)


# ============================================================
# Paths
# ============================================================

PROJECT_ROOT = config["PROJECT_ROOT"]
AILIB_ROOT = config["AILIB_ROOT"]

DATA_ROOT = config["DATA_ROOT"]
PROCESSED_DATA_DIR = config["PROCESSED_DATA_DIR"]
PROCESSED_DATA_ZIP = config["PROCESSED_DATA_ZIP"]

TRAIN_DIR = config["TRAIN_DIR"]
VAL_DIR = config["VAL_DIR"]
TEST_DIR = config["TEST_DIR"]

IMAGE_INFO_CSV = config["IMAGE_INFO_CSV"]

LOCAL_TRAIN_DIR = config["LOCAL_TRAIN_DIR"]
LOCAL_VAL_DIR = config["LOCAL_VAL_DIR"]
LOCAL_TEST_DIR = config["LOCAL_TEST_DIR"]

LOCAL_DATA_ROOT = config["LOCAL_DATA_ROOT"]
LOCAL_PROCESSED_DATA_DIR = config["LOCAL_PROCESSED_DATA_DIR"] 

DATASET_TRAIN_DIR = config["DATASET_TRAIN_DIR"]
DATASET_TEST_DIR = config["DATASET_TEST_DIR"]

EXPERIMENTS_DIR = config["EXPERIMENTS_DIR"]
MODELS_DIR = config["MODELS_DIR"]
REPORTS_DIR = config["REPORTS_DIR"]


# ============================================================
# Dataset Configuration
# ============================================================

CLASS_TO_IDX = config["CLASS_TO_IDX"]
IDX_TO_CLASS = config["IDX_TO_CLASS"]

In [5]:
print(PROCESSED_DATA_ZIP)
print(PROCESSED_DATA_ZIP.exists())

/content/drive/MyDrive/AI/projects/chest-xray-pneumonia/data/chest_xray_processed.zip
True


#### Reload

In [6]:
import importlib
import ailib.experiment

import cxp.cxp_config

importlib.reload(cxp.cxp_config)


<module 'cxp.cxp_config' from '/content/drive/MyDrive/AI/projects/chest-xray-pneumonia/cxp/cxp_config.py'>

### 1.2 Device Check

In [7]:
from ailib.device import (get_device, inspect_device )

device = get_device()
inspect_device() 

=== PyTorch Environment ===
PyTorch        : 2.11.0+cu128
Device         : cuda
CUDA Available : True
GPU            : Tesla T4


### 1.3 Resource Check

In [8]:
from ailib.resource import check_resource_safety

safe, reason = check_resource_safety(device)

print("Resource Safe:", safe)

if not safe:
    print("Reason:", reason)

Resource Safe: True


### 1.4 Project / Data Path  

In [9]:
print(DATA_ROOT)
print(PROCESSED_DATA_ZIP)

print(LOCAL_DATA_ROOT)
print(LOCAL_PROCESSED_DATA_DIR)


print("DATA_ROOT        :", DATA_ROOT)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("PROCESSED_DATA_ZIP:", PROCESSED_DATA_ZIP)


/content/drive/MyDrive/AI/projects/chest-xray-pneumonia/data
/content/drive/MyDrive/AI/projects/chest-xray-pneumonia/data/chest_xray_processed.zip
/content
/content/chest_xray_processed
DATA_ROOT        : /content/drive/MyDrive/AI/projects/chest-xray-pneumonia/data
PROCESSED_DATA_DIR: /content/drive/MyDrive/AI/projects/chest-xray-pneumonia/data/chest_xray_processed
PROCESSED_DATA_ZIP: /content/drive/MyDrive/AI/projects/chest-xray-pneumonia/data/chest_xray_processed.zip


### 1.5 Colab Context Load  
- Colab -> content에 전처리된 Process 이미지 Load
- 꼭!! zip파일로 드라이드에 업로드 후 context에 압축풀기 : GoogleDrive File I/O 이슈

In [10]:
import zipfile
 
with zipfile.ZipFile(PROCESSED_DATA_ZIP, "r") as z:
    z.extractall(LOCAL_DATA_ROOT)


print("Train:", (LOCAL_PROCESSED_DATA_DIR / "train").exists())
print("Val  :", (LOCAL_PROCESSED_DATA_DIR / "val").exists())
print("Test :", (LOCAL_PROCESSED_DATA_DIR / "test").exists()) 

Train: True
Val  : True
Test : True


In [34]:
print(list(LOCAL_DATA_ROOT.iterdir()))
for p in LOCAL_PROCESSED_DATA_DIR.rglob("*"):
    if p.is_dir():
        print(p)

[PosixPath('/content/.config'), PosixPath('/content/chest_xray_processed'), PosixPath('/content/drive'), PosixPath('/content/sample_data')]
/content/chest_xray_processed/val
/content/chest_xray_processed/train
/content/chest_xray_processed/test
/content/chest_xray_processed/val/NORMAL
/content/chest_xray_processed/val/PNEUMONIA
/content/chest_xray_processed/train/NORMAL
/content/chest_xray_processed/train/PNEUMONIA
/content/chest_xray_processed/test/NORMAL
/content/chest_xray_processed/test/PNEUMONIA


## 2. Data

### 2.1 Load Image Information

In [11]:
# 전처리된 이미지의 MetaData

from cxp.cxp_data import load_image_info

image_info = load_image_info(IMAGE_INFO_CSV) 

print("Shape:", image_info.shape) 
print("Columns:")
print(image_info.columns.tolist())
print()
print("Class:")
print(image_info["Folder"].value_counts())
 

Shape: (5216, 16)
Columns:
['Folder', 'File', 'Format', 'Width', 'Height', 'Shape', 'Channels', 'Mode', 'DType', 'File Size (KB)', 'Missing', 'Min', 'Max', 'Mean', 'Std', 'Error']

Class:
Folder
PNEUMONIA    3875
NORMAL       1341
Name: count, dtype: int64


### 2.2 Global Train Pixel Mean / Std (Pretraining model에서는 사용안함)

In [12]:
 
# ============================================================
# Global Train Pixel Mean / Std
# ============================================================

# 전체 Train 픽셀 평균
global_mean = image_info["Mean"].mean()


# 전체 Train 픽셀 분산
global_var = (
    (
        image_info["Std"] ** 2
        + (image_info["Mean"] - global_mean) ** 2
    ).mean()
)


# 전체 Train 픽셀 표준편차
global_std = global_var ** 0.5


# ToTensor() 이후 0~1 범위로 변환
mean = global_mean / 255
std = global_std / 255


print(f"Mean: {mean:.6f}")
print(f"Std : {std:.6f}") 

Mean: 0.345086
Std : 0.297072


### 2.3 Train / Validation Split(V2)

In [13]:
# ============================================================
#  Train / Validation Split
# ============================================================

from ailib.data_split_v2 import split_train_validation


train_df, val_df = split_train_validation(
    dataframe=image_info,
    val_size=0.2,
    random_state=42,
    stratify_column="Folder",
)


print("Train     :", len(train_df))
print("Validation:", len(val_df))

print()

print("Train:")
print(train_df["Folder"].value_counts())

print()

print("Validation:")
print(val_df["Folder"].value_counts())

Train     : 4172
Validation: 1044

Train:
Folder
PNEUMONIA    3099
NORMAL       1073
Name: count, dtype: int64

Validation:
Folder
PNEUMONIA    776
NORMAL       268
Name: count, dtype: int64


### 2.4 Test DataFrame(V2)

In [14]:
# ============================================================
#  Test DataFrame
# ============================================================

test_rows = []

for folder in CLASS_TO_IDX.keys():

    folder_path = DATASET_TEST_DIR / folder

    for file_path in folder_path.iterdir():

        if file_path.is_file():

            test_rows.append({
                "Folder": folder,
                "File": file_path.name,
            })


test_df = pd.DataFrame(test_rows)


print("Test:", len(test_df))
print()

print(test_df["Folder"].value_counts())

Test: 624

Folder
PNEUMONIA    390
NORMAL       234
Name: count, dtype: int64


### 2.5 Normalization(V2)

In [15]:
    # ============================================================
    #  Normalization
    # ============================================================

    NORMALIZATION = {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
    }


    print("Normalization")
    print("Mean:", NORMALIZATION["mean"])
    print("Std :", NORMALIZATION["std"])

Normalization
Mean: [0.485, 0.456, 0.406]
Std : [0.229, 0.224, 0.225]


### 2.6 Transform 생성(V2)

In [16]:
# ============================================================
#  Transform
# ============================================================

from ailib.transform_v2 import create_transform

IMAGE_SIZE = 512

train_transform = create_transform(
    image_size=IMAGE_SIZE,
    mean=NORMALIZATION["mean"],
    std=NORMALIZATION["std"],
)


eval_transform = create_transform(
    image_size=IMAGE_SIZE,
    mean=NORMALIZATION["mean"],
    std=NORMALIZATION["std"],
)


print("Train Transform:")
print(train_transform)

print()

print("Eval Transform:")
print(eval_transform)

Train Transform:
Compose(
    Resize(size=(512, 512), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Eval Transform:
Compose(
    Resize(size=(512, 512), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [17]:
# ============================================================
# Dataset Split Metadata
# ============================================================

import json
from datetime import datetime, timezone, timedelta


# ============================================================
# Split Configuration
# ============================================================

SPLIT_VERSION = "1.0"

VAL_SIZE = 0.2
RANDOM_STATE = 42
STRATIFY_COLUMN = "Folder"

SOURCE_DATASET = "chest-xray-pneumonia"


# ============================================================
# Path
# ============================================================

# reports 폴더가 없으면 생성
REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

split_config_path = REPORTS_DIR / "split_config.json"


# ============================================================
# Timestamp
# ============================================================

KST = timezone(timedelta(hours=9))

now = datetime.now(KST).isoformat(
    timespec="seconds"
)


# ============================================================
# Load Existing Metadata
# ============================================================

existing_metadata = None

if split_config_path.exists():

    with open(
        split_config_path,
        "r",
        encoding="utf-8",
    ) as f:

        existing_metadata = json.load(f)


# ============================================================
# Create / Update Metadata
# ============================================================

if existing_metadata is None:

    # --------------------------------------------------------
    # New
    # --------------------------------------------------------

    created_at = now
    updated_at = now

    created_by = None
    updated_by = None

    action = "CREATE"

else:

    # --------------------------------------------------------
    # Existing
    # --------------------------------------------------------

    if existing_metadata.get("version") == SPLIT_VERSION:

        # Same version → Update

        created_at = existing_metadata.get(
            "created_at",
            now,
        )

        created_by = existing_metadata.get(
            "created_by",
            None,
        )

        updated_at = now
        updated_by = None

        action = "UPDATE"

    else:

        # Different version → New Version

        created_at = now
        updated_at = now

        created_by = None
        updated_by = None

        action = "NEW VERSION"


# ============================================================
# Split Metadata
# ============================================================

split_metadata = {

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    "version": SPLIT_VERSION,

    "created_at": created_at,
    "updated_at": updated_at,

    "created_by": created_by,
    "updated_by": updated_by,


    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    "split_type": "train_validation",

    "source_dataset": SOURCE_DATASET,

    "source_metadata": str(
        IMAGE_INFO_CSV
    ),


    # --------------------------------------------------------
    # Validation Split
    # --------------------------------------------------------

    "validation": {

        "size": VAL_SIZE,

        "random_state": RANDOM_STATE,

        "stratify_column": STRATIFY_COLUMN,

    },


    # --------------------------------------------------------
    # Dataset Size
    # --------------------------------------------------------

    "dataset_size": {

        "total": len(image_info),

        "train": len(train_df),

        "validation": len(val_df),

        "test": len(test_df),

    },


    # --------------------------------------------------------
    # Class Distribution
    # --------------------------------------------------------

    "class_distribution": {

        "train": (
            train_df[STRATIFY_COLUMN]
            .value_counts()
            .to_dict()
        ),

        "validation": (
            val_df[STRATIFY_COLUMN]
            .value_counts()
            .to_dict()
        ),

        "test": (
            test_df[STRATIFY_COLUMN]
            .value_counts()
            .to_dict()
        ),

    },


    # --------------------------------------------------------
    # Preprocessing
    # --------------------------------------------------------

    "preprocessing": {

        "image_size": IMAGE_SIZE,

        "normalization": NORMALIZATION,

    },

}


# ============================================================
# Save
# ============================================================

with open(
    split_config_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        split_metadata,
        f,
        indent=4,
        ensure_ascii=False,
    )


# ============================================================
# Result
# ============================================================

print("=" * 60)
print("Split Metadata")
print("=" * 60)

print(f"Action  : {action}")
print(f"Version : {SPLIT_VERSION}")
print(f"Created : {created_at}")
print(f"Updated : {updated_at}")
print(f"File    : {split_config_path}")
print("=" * 60)

Split Metadata
Action  : UPDATE
Version : 1.0
Created : 2026-09-24T09:24:50+09:00
Updated : 2026-09-25T10:43:55+09:00
File    : /content/drive/MyDrive/AI/projects/chest-xray-pneumonia/reports/split_config.json


### 2.7 Dataset(V2)

In [18]:
# ============================================================
#  Dataset
# ============================================================

from ailib.dataset_v2 import ImageDataset


train_dataset = ImageDataset(
    dataframe=train_df,
    data_dir=DATASET_TRAIN_DIR,
    class_to_idx=CLASS_TO_IDX,
    transform=train_transform,
)


val_dataset = ImageDataset(
    dataframe=val_df,
    data_dir=DATASET_TRAIN_DIR,
    class_to_idx=CLASS_TO_IDX,
    transform=eval_transform,
)


test_dataset = ImageDataset(
    dataframe=test_df,
    data_dir=DATASET_TEST_DIR,
    class_to_idx=CLASS_TO_IDX,
    transform=eval_transform,
)


print("Train Dataset :", len(train_dataset))
print("Val Dataset   :", len(val_dataset))
print("Test Dataset  :", len(test_dataset))

Train Dataset : 4172
Val Dataset   : 1044
Test Dataset  : 624


In [19]:
# ============================================================
# STEP 6. Dataset Check
# ============================================================

image, label = train_dataset[0]


print("Image shape :", image.shape)
print("Image dtype :", image.dtype)
print("Label       :", label)
print("Label type  :", type(label))

print(
    "Image finite:",
    image.isfinite().all().item()
)

print(
    "Image min   :",
    image.min().item()
)

print(
    "Image max   :",
    image.max().item()
)

print(
    "Image mean  :",
    image.mean().item()
)

print(
    "Image std   :",
    image.std().item()
)

Image shape : torch.Size([3, 512, 512])
Image dtype : torch.float32
Label       : 0
Label type  : <class 'int'>
Image finite: True
Image min   : -2.1179039478302
Image max   : 2.640000104904175
Image mean  : -0.564913809299469
Image std   : 1.2356187105178833


### 2.8 Training Configuration(V2)

In [20]:
# ============================================================
#  Training Configuration
# ============================================================

LEARNING_RATE = 0.001  #Default


if ENV == "pc":
    BATCH_SIZE = 16

elif ENV == "remote":
    BATCH_SIZE = 64

elif ENV == "colab":
    BATCH_SIZE = 64

else:
    raise ValueError(
        f"Unknown environment: {ENV}"
    )


print("Learning Rate:", LEARNING_RATE)
print("Batch Size   :", BATCH_SIZE)

Learning Rate: 0.001
Batch Size   : 64


### 2.9 DataLoader(V2)

In [21]:
# ============================================================
#   DataLoader
# ============================================================

from ailib.dataloader_v2 import create_dataloader


train_loader = create_dataloader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

val_loader = create_dataloader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = create_dataloader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)


print("Train Loader :", len(train_loader))
print("Val Loader   :", len(val_loader))
print("Test Loader  :", len(test_loader))

Train Loader : 65
Val Loader   : 17
Test Loader  : 10


In [22]:
# ============================================================
# STEP 9. DataLoader Final Check
# ============================================================

from ailib.dataloader_v2 import inspect_dataloader


train_check = inspect_dataloader(
    train_loader,
    name="train",
)

val_check = inspect_dataloader(
    val_loader,
    name="validation",
)

test_check = inspect_dataloader(
    test_loader,
    name="test",
)


print("Train:")
print(train_check)

print()
print("Validation:")
print(val_check)

print()
print("Test:")
print(test_check)

Train:
{'name': 'train', 'batch_size': 64, 'image_shape': (64, 3, 512, 512), 'image_dtype': 'torch.float32', 'image_min': -2.1179039478302, 'image_max': 2.640000104904175, 'label_shape': (64,), 'label_dtype': 'torch.int64'}

Validation:
{'name': 'validation', 'batch_size': 64, 'image_shape': (64, 3, 512, 512), 'image_dtype': 'torch.float32', 'image_min': -2.1179039478302, 'image_max': 2.640000104904175, 'label_shape': (64,), 'label_dtype': 'torch.int64'}

Test:
{'name': 'test', 'batch_size': 64, 'image_shape': (64, 3, 512, 512), 'image_dtype': 'torch.float32', 'image_min': -2.1179039478302, 'image_max': 2.640000104904175, 'label_shape': (64,), 'label_dtype': 'torch.int64'}


## 3. Experiment Configuration

### 3.1 Model Configuration

In [23]:
# ============================================================
# 3.1 Model Configuration
# ============================================================


NUM_CLASSES = 2
IN_CHANNELS = 3

MODEL_CONFIGS = {
    "resnet50": {
        "model_name": "resnet50",
        "num_classes": NUM_CLASSES,
        "in_channels": IN_CHANNELS,
        "pretrained": True,
    },
    "densenet121": {
        "model_name": "densenet121",
        "num_classes": NUM_CLASSES,
        "in_channels": IN_CHANNELS,
        "pretrained": True,
    },
    "efficientnet_b0": {
        "model_name": "efficientnet_b0",
        "num_classes": NUM_CLASSES,
        "in_channels": IN_CHANNELS,
        "pretrained": True,
    },
}
 

# 실험에서 사용할 모델
# MODEL_NAME = "resnet50"
MODEL_NAME  = "densenet121"  ## 두번째 모델


MODEL_CONFIG = MODEL_CONFIGS[MODEL_NAME]


print(MODEL_CONFIG) 


{'model_name': 'densenet121', 'num_classes': 2, 'in_channels': 3, 'pretrained': True}


### 3.2 Training Configuration

In [24]:
# ============================================================
# 3.2 Training Configuration
# ============================================================

EPOCHS = 10   ##TEMP

LEARNING_RATES = {
    "LR1": 1e-3,
    "LR2": 1e-4,
    "LR3": 1e-5,
}

LEARNING_RATE_NAME = "LR1"
LEARNING_RATE = LEARNING_RATES[LEARNING_RATE_NAME]

WEIGHT_DECAY = 0.0

OPTIMIZER_NAME = "Adam"

LOSS_NAME = "CrossEntropyLoss"

TRAIN_CONFIG = {
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "optimizer": OPTIMIZER_NAME,
    "loss": LOSS_NAME,
}

print(TRAIN_CONFIG)

{'batch_size': 64, 'epochs': 10, 'learning_rate': 0.001, 'weight_decay': 0.0, 'optimizer': 'Adam', 'loss': 'CrossEntropyLoss'}


### 3.3 Fine-Tuning Configuration

In [25]:
# ============================================================
# 3.3 Fine-Tuning Configuration
# ============================================================

FINE_TUNING_MODES = [
    "frozen",
    "partial",
    "full",
]
 


FINE_TUNING_CONFIG = {
    "frozen": {
        "description": "Freeze backbone, train classification head",
    },
    "partial": {
        "description": "Train selected final backbone layers and classification head",
    },
    "full": {
        "description": "Train entire model",
    },
}


print("Fine-Tuning Modes")

for mode in FINE_TUNING_MODES:
    print(
        f"- {mode}: "
        f"{FINE_TUNING_CONFIG[mode]['description']}"
    )

# print(f"\nCurrent Fine-Tuning Mode: {FINE_TUNING_MODE}")

Fine-Tuning Modes
- frozen: Freeze backbone, train classification head
- partial: Train selected final backbone layers and classification head
- full: Train entire model


## 4. Transfer Learning Model

### 4.1 Load Pretrained ResNet18 -> densenet121

In [26]:
# ============================================================
# 4.1 Load Pretrained Model
# ============================================================

from torchvision import models


MODEL_NAME  = "densenet121"  ## 두번째 모델 ( 모델이 변경될때 꼭!! 변경 )



if MODEL_NAME == "resnet50":
    model = models.resnet50(
        weights=models.ResNet50_Weights.DEFAULT
    )

elif MODEL_NAME == "densenet121":
    model = models.densenet121(
        weights=models.DenseNet121_Weights.DEFAULT
    )

elif MODEL_NAME == "efficientnet_b0":
    model = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.DEFAULT
    )

else:
    raise ValueError(
        f"Unsupported model: {MODEL_NAME}"
    )

print(f"Loaded pretrained model: {MODEL_NAME}")

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 227MB/s]


Loaded pretrained model: densenet121


### 4.2 Adapt Input Layer(Deprecated : 채널을 1 -> 3d 으로 변경해서 4.2 실행안함)

In [ ]:
# # ============================================================
# # 4.2 Adapt Input Layer
# # ============================================================

# import torch
# import torch.nn as nn


# if MODEL_NAME == "resnet50":

#     old_conv = model.conv1

#     model.conv1 = nn.Conv2d(
#         in_channels=IN_CHANNELS,
#         out_channels=old_conv.out_channels,
#         kernel_size=old_conv.kernel_size,
#         stride=old_conv.stride,
#         padding=old_conv.padding,
#         bias=False,
#     )

#     with torch.no_grad():
#         model.conv1.weight.copy_(
#             old_conv.weight.mean(
#                 dim=1,
#                 keepdim=True
#             )
#         )


# elif MODEL_NAME == "densenet121":

#     old_conv = model.features.conv0

#     model.features.conv0 = nn.Conv2d(
#         in_channels=IN_CHANNELS,
#         out_channels=old_conv.out_channels,
#         kernel_size=old_conv.kernel_size,
#         stride=old_conv.stride,
#         padding=old_conv.padding,
#         bias=False,
#     )

#     with torch.no_grad():
#         model.features.conv0.weight.copy_(
#             old_conv.weight.mean(
#                 dim=1,
#                 keepdim=True
#             )
#         )


# elif MODEL_NAME == "efficientnet_b0":

#     old_conv = model.features[0][0]

#     model.features[0][0] = nn.Conv2d(
#         in_channels=IN_CHANNELS,
#         out_channels=old_conv.out_channels,
#         kernel_size=old_conv.kernel_size,
#         stride=old_conv.stride,
#         padding=old_conv.padding,
#         bias=False,
#     )

#     with torch.no_grad():
#         model.features[0][0].weight.copy_(
#             old_conv.weight.mean(
#                 dim=1,
#                 keepdim=True
#             )
#         )


# print(f"Input channels: {IN_CHANNELS}")

### 4.3 Replace Classification Head

In [27]:
# ============================================================
# 4.3 Replace Classification Head
# ============================================================

import torch.nn as nn

if MODEL_NAME == "resnet50":

    model.fc = nn.Linear(
        model.fc.in_features,
        NUM_CLASSES,
    )


elif MODEL_NAME == "densenet121":

    model.classifier = nn.Linear(
        model.classifier.in_features,
        NUM_CLASSES,
    )


elif MODEL_NAME == "efficientnet_b0":

    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        NUM_CLASSES,
    )


print(f"Output classes: {NUM_CLASSES}")

Output classes: 2


### 4.4 Freeze / Unfreeze Configuration

In [28]:
# ============================================================
# 4.4 Freeze / Unfreeze Configuration
# ============================================================

def configure_fine_tuning(
    model,
    model_name,
    fine_tuning_mode,
):
    """
    Configure trainable parameters for:
        - frozen
        - partial
        - full
    """

    # --------------------------------------------------------
    # 1. Freeze everything
    # --------------------------------------------------------
    for parameter in model.parameters():
        parameter.requires_grad = False

    # --------------------------------------------------------
    # 2. Frozen
    #    → classification head only
    # --------------------------------------------------------
    if fine_tuning_mode == "frozen":

        if model_name == "resnet50":
            for parameter in model.fc.parameters():
                parameter.requires_grad = True

        elif model_name == "densenet121":
            for parameter in model.classifier.parameters():
                parameter.requires_grad = True

        elif model_name == "efficientnet_b0":
            for parameter in model.classifier.parameters():
                parameter.requires_grad = True

    # --------------------------------------------------------
    # 3. Partial
    # --------------------------------------------------------
    elif fine_tuning_mode == "partial":

        if model_name == "resnet50":

            for parameter in model.layer4.parameters():
                parameter.requires_grad = True

            for parameter in model.fc.parameters():
                parameter.requires_grad = True

        elif model_name == "densenet121":

            for parameter in model.features.denseblock4.parameters():
                parameter.requires_grad = True

            for parameter in model.features.norm5.parameters():
                parameter.requires_grad = True

            for parameter in model.classifier.parameters():
                parameter.requires_grad = True

        elif model_name == "efficientnet_b0":

            for parameter in model.features[-1].parameters():
                parameter.requires_grad = True

            for parameter in model.classifier.parameters():
                parameter.requires_grad = True

    # --------------------------------------------------------
    # 4. Full
    # --------------------------------------------------------
    elif fine_tuning_mode == "full":

        for parameter in model.parameters():
            parameter.requires_grad = True

    else:
        raise ValueError(
            f"Unsupported fine-tuning mode: "
            f"{fine_tuning_mode}"
        )

    return model

### 4.5 Trainable Parameter Check

In [29]:
# ============================================================
# 4.5 Trainable Parameter Check
# ============================================================

def check_trainable_parameters(model):

    total_parameters = 0
    trainable_parameters = 0

    for parameter in model.parameters():

        num_parameters = parameter.numel()

        total_parameters += num_parameters

        if parameter.requires_grad:
            trainable_parameters += num_parameters

    frozen_parameters = (
        total_parameters
        - trainable_parameters
    )

    trainable_ratio = (
        trainable_parameters / total_parameters * 100
    )

    print("Parameter Check")
    print("-" * 40)
    print(f"Total      : {total_parameters:,}")
    print(f"Trainable  : {trainable_parameters:,}")
    print(f"Frozen     : {frozen_parameters:,}")
    print(f"Trainable %: {trainable_ratio:.2f}%")

    return {
        "total": total_parameters,
        "trainable": trainable_parameters,
        "frozen": frozen_parameters,
        "trainable_ratio": trainable_ratio,
    }

### 4.6 Common Training Function

In [30]:
# ============================================================
# Common Training Function
# ============================================================

from pathlib import Path

from ailib.experiment import save_experiment

from cxp.cxp_model import (
    build_transfer_model,
    configure_fine_tuning,
    check_trainable_parameters,
)

from cxp.cxp_training import run_experiment


# ============================================================
# Transfer Learning Experiment
# ============================================================

def run_transfer_learning_experiment(
    experiment_name,
    model_name,
    fine_tuning_mode,
    experiment_learning_rates,
    train_loader,
    val_loader,
    epochs,
):
    """
    Transfer Learning 실험 공통 Training 함수.

    Each learning-rate run is stored independently.

    Checkpoint behavior
    -------------------
    - checkpoint가 없으면 새로 학습
    - checkpoint가 있으면 마지막 완료 epoch부터 resume
    - epochs는 전체 목표 epoch 수
    """

    experiment_results = {}

    for lr_name, learning_rate in experiment_learning_rates.items():

        print("=" * 60)
        print(f"Experiment: {experiment_name}")
        print(
            f"Learning Rate: "
            f"{lr_name} = {learning_rate}"
        )
        print(f"Model: {model_name}")
        print(
            f"Fine-Tuning: "
            f"{fine_tuning_mode}"
        )
        print(f"Target Epochs: {epochs}")
        print("=" * 60)

        # ----------------------------------------------------
        # Model Configuration
        # ----------------------------------------------------

        model_config = MODEL_CONFIGS[model_name]

        # ----------------------------------------------------
        # Fresh Pretrained Model
        # ----------------------------------------------------

        model = build_transfer_model(
            model_name=model_config["model_name"],
            num_classes=model_config["num_classes"],
            in_channels=model_config["in_channels"],
            pretrained=model_config["pretrained"],
        )

        # ----------------------------------------------------
        # Fine-Tuning Configuration
        # ----------------------------------------------------

        model = configure_fine_tuning(
            model=model,
            model_name=model_name,
            fine_tuning_mode=fine_tuning_mode,
        )

        # ----------------------------------------------------
        # Trainable Parameter Check
        # ----------------------------------------------------

        parameter_info = check_trainable_parameters(
            model
        )

        print(
            f"Trainable Parameters: "
            f"{parameter_info['trainable_parameters']:,}"
        )

        print(
            f"Trainable Ratio: "
            f"{parameter_info['trainable_ratio']:.2%}"
        )

        # ----------------------------------------------------
        # Experiment Name
        # ----------------------------------------------------

        current_experiment_name = (
            f"{experiment_name}/{lr_name}"
        )

        # ----------------------------------------------------
        # Checkpoint Directory
        # ----------------------------------------------------

        checkpoint_dir = (
            Path(EXPERIMENTS_DIR)
            / current_experiment_name
        )

        print(
            f"Checkpoint Directory: "
            f"{checkpoint_dir}"
        )

        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        (
            model,
            config,
            history,
            result,
        ) = run_experiment(
            experiment_name=current_experiment_name,
            optimizer_name=OPTIMIZER_NAME,
            num_epochs=epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            learning_rate=learning_rate,
            weight_decay=WEIGHT_DECAY,
            model=model,
            image_size=512,
            channels=IN_CHANNELS,
            normalization="imagenet_mean_std",
            augmentation="none",
            loss_name="CrossEntropy",
            checkpoint_dir=checkpoint_dir,
        )

        # ----------------------------------------------------
        # Save Final Experiment
        # ----------------------------------------------------

        save_experiment(
            root_dir=EXPERIMENTS_DIR,
            experiment_name=current_experiment_name,
            model=model,
            config=config,
            history=history,
            result=result,
            save_model=True,
        )

        # ----------------------------------------------------
        # Keep Lightweight In-Memory Results
        # ----------------------------------------------------

        experiment_results[lr_name] = {
            "config": config,
            "history": history,
            "result": result,
        }

        print(
            f"Completed: "
            f"{current_experiment_name}"
        )

    return experiment_results

## 5. Experiment 1 — Learning Rate Effect

### 5.1 Configuration

In [31]:
# ============================================================
# 5.1 Configuration
# ============================================================

EXPERIMENT_NAME = "TL-RESNET50-FROZEN-LR"

MODEL_NAME = "resnet50"
FINE_TUNING_MODE = "frozen"

EXPERIMENT_LEARNING_RATES = LEARNING_RATES

print("Experiment:", EXPERIMENT_NAME)
print("Model:", MODEL_NAME)
print("Fine-Tuning:", FINE_TUNING_MODE)
print("Learning Rates:", EXPERIMENT_LEARNING_RATES) 


Experiment: TL-RESNET50-FROZEN-LR
Model: resnet50
Fine-Tuning: frozen
Learning Rates: {'LR1': 0.001, 'LR2': 0.0001, 'LR3': 1e-05}


In [ ]:
# ============================================================
# 5.1-E Configuration
# ============================================================

EXPERIMENT_NAME = "TL-RESNET50-FROZEN-LR"

MODEL_NAME = "resnet50"
FINE_TUNING_MODE = "frozen"

EXPERIMENT_LEARNING_RATES = {
    "LR2-EPOCH30": 1e-4,
}

print("Experiment:", EXPERIMENT_NAME)
print("Model:", MODEL_NAME)
print("Fine-Tuning:", FINE_TUNING_MODE)
print("Learning Rates:", EXPERIMENT_LEARNING_RATES) 


Experiment: TL-RESNET50-FROZEN-LR
Model: resnet50
Fine-Tuning: frozen
Learning Rates: {'LR2-EPOCH30': 0.0001}


### 5.2 Training

#### **** SourceReload

In [ ]:
 
# ============================================================
# Source Reload
# ============================================================

import importlib

import cxp.cxp_training as cxp_training
import cxp.cxp_model as cxp_model 


importlib.reload(cxp_training)
importlib.reload(cxp_model) 
 
from cxp.cxp_model import ( build_transfer_model, configure_fine_tuning, check_trainable_parameters, )
print("cxp_training.py reloaded.") 

 
# ============================================================
# Experiment Source Reload
# ============================================================

import importlib

import ailib.experiment as experiment

importlib.reload(experiment) 

print("ailib.experiment.py reloaded.") 



cxp_training.py reloaded.
ailib.experiment.py reloaded.


#### ResNet50 NaN Check (->)

In [ ]:
# ============================================================
# Experiment Run - Cleanup & Environment Check
# ============================================================

import gc
import torch

print("=" * 50)
print("Experiment Environment Check")

# Python / CUDA memory cleanup
gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    print(f"Device : CUDA")
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(
        f"Memory : "
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB allocated / "
        f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB reserved"
    )

else:
    print("Device : CPU")

print("=" * 50)
print("Ready for Experiment")
print("=" * 50)

Experiment Environment Check
Device : CUDA
GPU    : NVIDIA GeForce RTX 5070
Memory : 0.00 GB allocated / 0.00 GB reserved
Ready for Experiment


### **  실행 일괄 삭제 ** : 삭제가 필요한 경우에만 

In [ ]:

# # ============================================================
# # Experiment Cleanup
# # 잘못 실행되거나 중단된 실험 정리용
# # ============================================================

# import shutil
# from pathlib import Path

# # 실행 결과 및 임시 객체 초기화
# for name in [
#     "experiment_results",
#     "model",
#     "pipeline",
#     "result",
#     "validation_results",
#     "validation_metrics",
#     "validation_df",
#     "validation_metrics_df",
# ]:
#     globals().pop(name, None)

# # Experiment 결과 폴더 삭제
# experiment_root = (
#     Path(EXPERIMENTS_DIR)
#     / "TL-RESNET50-FROZEN-LR"
# )

# if experiment_root.exists():

#     print(f"Delete: {experiment_root}")

#     if input("Continue? [y/N]: ").lower() == "y":
#         shutil.rmtree(experiment_root)
#         print("Deleted.")

#     else:
#         print("Cancelled.")

# else:
#     print("No experiment directory found.") 

## 6. Experiment 2 — Partial Fine-Tuning

### 6.1 Configuration

In [ ]:
# ============================================================
# 6.1 Configuration
# Experiment 2 - Partial Fine-Tuning-E20
# ============================================================

EXPERIMENT_NAME = "TL-RESNET50-PARTIAL-E20"

MODEL_NAME = "resnet50"
FINE_TUNING_MODE = "partial"

EXPERIMENT_LEARNING_RATES = {
    "LR1": 1e-4,
}


print("Experiment:", EXPERIMENT_NAME)
print("Model:", MODEL_NAME)
print("Fine-Tuning:", FINE_TUNING_MODE)
print("Learning Rates:", EXPERIMENT_LEARNING_RATES)

Experiment: TL-RESNET50-PARTIAL-E20
Model: resnet50
Fine-Tuning: partial
Learning Rates: {'LR1': 0.0001}


### 6.2 Training

In [ ]:
# ============================================================
# 6.3 Training
# ============================================================

experiment_results = run_transfer_learning_experiment(
    experiment_name=EXPERIMENT_NAME,
    model_name=MODEL_NAME,
    fine_tuning_mode=FINE_TUNING_MODE,
    experiment_learning_rates=EXPERIMENT_LEARNING_RATES,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
)

Experiment: TL-RESNET50-PARTIAL-E20
Learning Rate: LR1 = 0.0001
Model: resnet50
Fine-Tuning: partial
Epochs: 20
Trainable Parameters: 14,968,834
Trainable Ratio: 63.66%
Device: cuda
AMP: OFF
[Batch 1/65] Data Load: 2.104s | Model: 0.472s | Total: 2.576s | GPU Alloc: 0.74GB | GPU Reserved: 4.32GB
[Train Epoch] Time: 150.49s | Loss: 0.2241 | Accuracy: 0.8954
[Val Batch 1/17] Data Load: 2.008s | Model: 0.282s | Total: 2.289s | GPU Alloc: 0.74GB | GPU Reserved: 6.32GB
[Val Batch 10/17] Data Load: 1.863s | Model: 0.285s | Total: 2.148s | GPU Alloc: 0.74GB | GPU Reserved: 6.32GB
[Validation Epoch] Time: 35.31s | Loss: 0.1320 | Accuracy: 0.9646
Epoch [1/20] Train Loss: 0.2241 Train Acc: 0.8954 Val Loss: 0.1320 Val Acc: 0.9646 Time: 185.80s
[Batch 1/65] Data Load: 1.873s | Model: 0.358s | Total: 2.232s | GPU Alloc: 0.74GB | GPU Reserved: 6.32GB
[Train Epoch] Time: 144.38s | Loss: 0.0519 | Accuracy: 0.9868
[Val Batch 1/17] Data Load: 1.797s | Model: 0.279s | Total: 2.076s | GPU Alloc: 0.74GB | 

## 7. Experiment 3 — Full Fine-Tuning

### 7.1 Configuration

In [ ]:
# ============================================================
# 7.1 Configuration
# Experiment 3 - Full Fine-Tuning
# ============================================================

EXPERIMENT_NAME = "TL-RESNET50-FULL-EP20"

MODEL_NAME = "resnet50"
FINE_TUNING_MODE = "full"

EXPERIMENT_LEARNING_RATES = {
    "LR1": 1e-4,
}

### 7.3 DataLoader

In [ ]:

# ============================================================
# 7.2 DataLoader
# Full Fine-Tuning - Batch Size 16
# ============================================================

from ailib.dataloader_v2 import create_dataloader

FULL_BATCH_SIZE = 16

full_train_loader = create_dataloader(
    dataset=train_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

full_val_loader = create_dataloader(
    dataset=val_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=False,
)
 
print("Full Batch Size   :", FULL_BATCH_SIZE)
print("Full Train batches:", len(full_train_loader))
print("Full Val batches  :", len(full_val_loader))

Full Batch Size   : 16
Full Train batches: 65
Full Val batches  : 17


### 7.4 Training

In [ ]:
# ============================================================
# 7.4 Training
# ============================================================

experiment_results = run_transfer_learning_experiment(
    experiment_name=EXPERIMENT_NAME,
    model_name=MODEL_NAME,
    fine_tuning_mode=FINE_TUNING_MODE,
    experiment_learning_rates=EXPERIMENT_LEARNING_RATES,
    train_loader=full_train_loader,
    val_loader=full_val_loader,
    epochs=EPOCHS,
)

Experiment: TL-RESNET50-FULL-EP20
Learning Rate: LR1 = 0.0001
Model: resnet50
Fine-Tuning: full
Epochs: 20
Trainable Parameters: 23,512,130
Trainable Ratio: 100.00%
Device: cuda
AMP: OFF
[Batch 1/260] Data Load: 0.493s | Model: 0.840s | Total: 1.333s | GPU Alloc: 0.78GB | GPU Reserved: 8.56GB
[Batch 100/260] Data Load: 0.450s | Model: 0.245s | Total: 0.696s | GPU Alloc: 0.70GB | GPU Reserved: 8.56GB
[Batch 200/260] Data Load: 0.436s | Model: 0.245s | Total: 0.681s | GPU Alloc: 0.70GB | GPU Reserved: 8.56GB
[Train Epoch] Time: 184.59s | Loss: 0.1310 | Accuracy: 0.9481
[Val Batch 1/66] Data Load: 0.434s | Model: 0.093s | Total: 0.527s | GPU Alloc: 0.70GB | GPU Reserved: 8.56GB
[Val Batch 10/66] Data Load: 0.447s | Model: 0.092s | Total: 0.540s | GPU Alloc: 0.70GB | GPU Reserved: 8.56GB
[Val Batch 20/66] Data Load: 0.508s | Model: 0.092s | Total: 0.600s | GPU Alloc: 0.70GB | GPU Reserved: 8.56GB
[Val Batch 30/66] Data Load: 0.559s | Model: 0.087s | Total: 0.646s | GPU Alloc: 0.70GB | GPU 

## 8. Experiment 4 — Densenet12 Learning Rate 

### 8.1 Configuration

In [49]:
# ============================================================
# 8.1 Configuration
# Experiment 4 - Densenet121 Learning Rate
# ============================================================

EXPERIMENT_NAME = "TL-DENSENET121-LR"

MODEL_NAME = "densenet121"
FINE_TUNING_MODE = "frozen"

EXPERIMENT_LEARNING_RATES = {
    "LR1": 1e-3,
    "LR2": 1e-4,
    # "LR3": 1e-5,  # Test결과 학습률 너무 떨어짐
}

EXPERIMENT_EPOCHS = 20


print("Experiment:", EXPERIMENT_NAME)
print("Model:", MODEL_NAME)
print("Fine-Tuning:", FINE_TUNING_MODE)
print("Learning Rates:", EXPERIMENT_LEARNING_RATES)

Experiment: TL-DENSENET121-LR
Model: densenet121
Fine-Tuning: frozen
Learning Rates: {'LR1': 0.001, 'LR2': 0.0001}


In [55]:
# ============================================================
# 8.2 Configuration
# Experiment 4 - Densenet121 Learning Rate- Epoch40
# ============================================================

EXPERIMENT_NAME = "TL-DENSENET121-LR-E40"

MODEL_NAME = "densenet121"
FINE_TUNING_MODE = "frozen"

EXPERIMENT_LEARNING_RATES = { 
    "LR2": 1e-4, 
}

EXPERIMENT_EPOCHS = 40


print("Experiment:", EXPERIMENT_NAME)
print("Model:", MODEL_NAME)
print("Fine-Tuning:", FINE_TUNING_MODE)
print("Learning Rates:", EXPERIMENT_LEARNING_RATES)
print("Epochs:", EXPERIMENT_EPOCHS)

Experiment: TL-DENSENET121-LR-E40
Model: densenet121
Fine-Tuning: frozen
Learning Rates: {'LR2': 0.0001}
Epochs: 40


### 8.2 Training

In [56]:
experiment_results = run_transfer_learning_experiment(
    experiment_name=EXPERIMENT_NAME,
    model_name=MODEL_NAME,
    fine_tuning_mode=FINE_TUNING_MODE,
    experiment_learning_rates=EXPERIMENT_LEARNING_RATES,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EXPERIMENT_EPOCHS,
)

Experiment: TL-DENSENET121-LR-E40
Learning Rate: LR2 = 0.0001
Model: densenet121
Fine-Tuning: frozen
Epochs: 40
Trainable Parameters: 2,050
Trainable Ratio: 0.03%
Device: cuda
AMP: OFF
[Batch 1/65] Data Load: 1.080s | Model: 0.485s | Total: 1.565s | GPU Alloc: 0.28GB | GPU Reserved: 6.67GB
[Train Epoch] Time: 85.05s | Loss: 0.5414 | Accuracy: 0.7406
[Val Batch 1/17] Data Load: 0.858s | Model: 0.408s | Total: 1.266s | GPU Alloc: 0.28GB | GPU Reserved: 6.67GB
[Val Batch 10/17] Data Load: 0.879s | Model: 0.406s | Total: 1.285s | GPU Alloc: 0.28GB | GPU Reserved: 6.67GB
[Validation Epoch] Time: 20.52s | Loss: 0.5005 | Accuracy: 0.7433
Epoch [1/40] Train Loss: 0.5414 Train Acc: 0.7406 Val Loss: 0.5005 Val Acc: 0.7433 Time: 105.58s
[Batch 1/65] Data Load: 0.781s | Model: 0.441s | Total: 1.222s | GPU Alloc: 0.28GB | GPU Reserved: 6.68GB
[Train Epoch] Time: 80.61s | Loss: 0.4604 | Accuracy: 0.7452
[Val Batch 1/17] Data Load: 0.796s | Model: 0.407s | Total: 1.203s | GPU Alloc: 0.28GB | GPU Rese

## 9. Experiment 5 — Densenet12 Partial Fine-Tuning 

### 9.1 Configuration

In [57]:
# ============================================================
# 9. Experiment 5 - DenseNet121 Partial Fine-Tuning
# ============================================================

EXPERIMENT_NAME = "TL-DENSENET121-PARTIAL"

MODEL_NAME = "densenet121"
FINE_TUNING_MODE = "partial"

EXPERIMENT_LEARNING_RATES = {
    "LR1": 1e-3,  # Experiment 4에서 선정
}

EXPERIMENT_EPOCHS = 20


print("Experiment:", EXPERIMENT_NAME)
print("Model:", MODEL_NAME)
print("Fine-Tuning:", FINE_TUNING_MODE)
print("Learning Rates:", EXPERIMENT_LEARNING_RATES)

Experiment: TL-DENSENET121-PARTIAL
Model: densenet121
Fine-Tuning: partial
Learning Rates: {'LR1': 0.001}


In [32]:
# ============================================================
# 9. Experiment 5-2 - DenseNet121 Partial Fine-Tuning
# ============================================================

EXPERIMENT_NAME = "TL-DENSENET121-PARTIAL"

MODEL_NAME = "densenet121"
FINE_TUNING_MODE = "partial"

EXPERIMENT_LEARNING_RATES = {
    "LR2": 1e-4,   
}

EXPERIMENT_EPOCHS = 20


print("Experiment:", EXPERIMENT_NAME)
print("Model:", MODEL_NAME)
print("Fine-Tuning:", FINE_TUNING_MODE)
print("Learning Rates:", EXPERIMENT_LEARNING_RATES)

Experiment: TL-DENSENET121-PARTIAL
Model: densenet121
Fine-Tuning: partial
Learning Rates: {'LR2': 0.0001}


### 9.2 Training

In [33]:
experiment_results = run_transfer_learning_experiment(
    experiment_name=EXPERIMENT_NAME,
    model_name=MODEL_NAME,
    fine_tuning_mode=FINE_TUNING_MODE,
    experiment_learning_rates=EXPERIMENT_LEARNING_RATES,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EXPERIMENT_EPOCHS,
)

Experiment: TL-DENSENET121-PARTIAL
Learning Rate: LR2 = 0.0001
Model: densenet121
Fine-Tuning: partial
Target Epochs: 20
Trainable Parameters: 2,162,178
Trainable Ratio: 31.08%
Checkpoint Directory: /content/drive/MyDrive/AI/projects/chest-xray-pneumonia/experiments/TL-DENSENET121-PARTIAL/LR2
Device: cuda
AMP: OFF
No checkpoint found. Starting from scratch.

[TL-DENSENET121-PARTIAL/LR2] Training range: Epoch 1 -> 20
[Batch 1/65] Data Load: 0.681s | Model: 4.014s | Total: 4.695s | GPU Alloc: 0.25GB | GPU Reserved: 6.02GB
[Batch 50/65] Data Load: 0.842s | Model: 1.127s | Total: 1.969s | GPU Alloc: 0.25GB | GPU Reserved: 6.02GB
[Train Epoch] Time: 120.01s | Loss: 0.1942 | Accuracy: 0.9305
[Val Batch 1/17] Data Load: 0.669s | Model: 0.941s | Total: 1.610s | GPU Alloc: 0.25GB | GPU Reserved: 6.02GB
[Val Batch 10/17] Data Load: 0.549s | Model: 0.950s | Total: 1.499s | GPU Alloc: 0.25GB | GPU Reserved: 6.02GB
[Validation Epoch] Time: 25.43s | Loss: 0.0887 | Accuracy: 0.9703
Checkpoint saved: 

## 10. Experiment 6 — Densenet12 Full Fine-Tuning 

### 10.1 Configuration

In [58]:
# ============================================================
# 10.1 Configuration
# Experiment 3 - Full Fine-Tuning
# ============================================================

EXPERIMENT_NAME = "TL-DENSENET121-FULL-EP20"

MODEL_NAME = "densenet121"
FINE_TUNING_MODE = "full"

EXPERIMENT_LEARNING_RATES = {
    "LR1": 1e-3,
}

EPOCHS = 20

In [62]:
# ============================================================
# 10.1 Configuration
# Experiment 3 - Full Fine-Tuning2
# ============================================================

EXPERIMENT_NAME = "TL-DENSENET121-FULL-EP20"

MODEL_NAME = "densenet121"
FINE_TUNING_MODE = "full"

EXPERIMENT_LEARNING_RATES = {
    "LR2": 1e-4,
}

EPOCHS = 20

In [59]:

# ============================================================
# 10.2 DataLoader
# Full Fine-Tuning - Batch Size 16
# ============================================================

from ailib.dataloader_v2 import create_dataloader

FULL_BATCH_SIZE = 16


full_train_loader = create_dataloader(
    dataset=train_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

full_val_loader = create_dataloader(
    dataset=val_dataset,
    batch_size=FULL_BATCH_SIZE,
    shuffle=False,
)
 
print("Full Batch Size   :", FULL_BATCH_SIZE)
print("Full Train batches:", len(full_train_loader))
print("Full Val batches  :", len(full_val_loader))

Full Batch Size   : 16
Full Train batches: 260
Full Val batches  : 66


In [60]:
print("Actual train batch size:", full_train_loader.batch_size)
print("Actual val batch size  :", full_val_loader.batch_size)
print("Train batches          :", len(full_train_loader))
print("Val batches            :", len(full_val_loader))

Actual train batch size: 16
Actual val batch size  : 16
Train batches          : 260
Val batches            : 66


### 10.2 Training

In [63]:
# ============================================================
# 10.2 Training
# ============================================================

experiment_results = run_transfer_learning_experiment(
    experiment_name=EXPERIMENT_NAME,
    model_name=MODEL_NAME,
    fine_tuning_mode=FINE_TUNING_MODE,
    experiment_learning_rates=EXPERIMENT_LEARNING_RATES,
    train_loader=full_train_loader,
    val_loader=full_val_loader,
    epochs=EPOCHS,
)

Experiment: TL-DENSENET121-FULL-EP20
Learning Rate: LR2 = 0.0001
Model: densenet121
Fine-Tuning: full
Target Epochs: 20
Trainable Parameters: 6,955,906
Trainable Ratio: 100.00%
Checkpoint Directory: /content/drive/MyDrive/AI/projects/chest-xray-pneumonia/experiments/TL-DENSENET121-FULL-EP20/LR2
Device: cuda
AMP: OFF
No checkpoint found. Starting from scratch.

[TL-DENSENET121-FULL-EP20/LR2] Training range: Epoch 1 -> 20
[Batch 1/260] Data Load: 0.169s | Model: 0.847s | Total: 1.016s | GPU Alloc: 0.17GB | GPU Reserved: 11.67GB
[Batch 50/260] Data Load: 0.218s | Model: 0.790s | Total: 1.007s | GPU Alloc: 0.17GB | GPU Reserved: 11.67GB
[Batch 100/260] Data Load: 0.185s | Model: 0.798s | Total: 0.983s | GPU Alloc: 0.17GB | GPU Reserved: 11.67GB
[Batch 150/260] Data Load: 0.209s | Model: 0.836s | Total: 1.045s | GPU Alloc: 0.17GB | GPU Reserved: 11.67GB
[Batch 200/260] Data Load: 0.170s | Model: 0.824s | Total: 0.994s | GPU Alloc: 0.17GB | GPU Reserved: 11.67GB
[Batch 250/260] Data Load: 0.

## 11. Conclusion


이번 실험에서는 동일한 데이터셋과 기본 학습 조건을 유지한 상태에서 **모델 구조, Learning Rate, Fine-tuning 방식**에 따른 학습 성능 변화를 비교하였다.

각 실험은 동일한 기준으로 학습을 수행하고, `config`, `history`, `model`, `result`를 Run 단위로 저장하여 이후 실험 결과를 다시 확인할 수 있도록 구성하였다.

이번 단계에서는 개별 실험의 **학습 과정과 Validation 성능을 확인**하는 데 집중하였다.

실험 결과는 이후 단계에서 모델별로 비교하고, 추가적인 평가 지표와 Error Analysis를 통해 모델의 성능과 특성을 확인한다.

### Next Step

- 실험별 Validation 결과 비교
- ROC-AUC / PR-AUC 등 추가 평가 지표 확인
- Confusion Matrix 및 Error Analysis
- Grad-CAM을 통한 모델의 이미지 판단 영역 확인
- 평가 결과를 바탕으로 추가 실험 여부 결정


### Analysis Notebook 
[03-0.Experiments MetaData](./analysis/00_cxp_analysis_meta.ipynb)<br>
→ 실험 설정과 Run 결과를 Metadata로 정리<br><br>

[03-1.ResNet50 LR Experiment Analysis](./analysis/01_cxp_resnet50_lr.ipynb)<br>
→ ResNet50 LR별 실험 결과 비교 및 Validation 평가<br>

[03-2.DenseNet121 LR Experiment Analysis](./analysis/02_cxp_densenet121_lr.ipynb)<br>
→ DenseNet121 LR별 실험 결과 비교 및 Validation 평가<br>

[03-3.ResNet50 FineTuning Experiment Analysis](./analysis/03_cxp_resnet50_finetuning.ipynb)<br>
→ ResNet50 FineTuning 실험 결과 비교 및 Validation 평가<br>

[03-4.DenseNet121 FineTuning Experiment Analysis](./analysis/04_cxp_densenet121_finetuning.ipynb)<br>
→ DenseNet121 FineTuning 실험 결과 비교 및 Validation 평가<br>

[03-5.ResNet50, DenseNet121 Comparison Analysis](./analysis/05_cxp_model_finetuning_comparison.ipynb)<br>
→ ResNet50,DenseNet121 FineTuning 비교 분석<br>
 